# Literature GAP Matrix via RAG — Domain Kesehatan
MK Introduction to Generative AI (P36104) · Magister AI UDINUS

Pipeline: **PyMuPDF → chunking → `paraphrase-multilingual-mpnet-base-v2` (lokal) → FAISS → retrieval top-k per kolom → Groq `openai/gpt-oss-120b` → gap matrix + analisis gap**

Kolom sesuai format dosen: *Paper | DOI | Problem | Metode | Dataset | Temuan | Keterbatasan | Future work*

**Sebelum mulai:** simpan API key Groq di Colab **Secrets** (ikon kunci di sidebar) dengan nama `GROQ_API_KEY`.

## 1. Clone repo & install

In [ ]:
!git clone https://github.com/bintangningwang/rag-gap-matrix-kesehatan.git
%cd rag-gap-matrix-kesehatan
!pip install -q -r requirements.txt

## 2. Konfigurasi

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

os.environ["GITHUB_URL"] = "https://github.com/bintangningwang/rag-gap-matrix-kesehatan"   # ganti
os.environ["NAMA_MAHASISWA"] = "Bintang Nugraha Kasaluri Satrianingwang"          # tambah anggota bila kelompok
# os.environ["TOP_K"] = "2"   # aktifkan jika muncul error "Prompt terlalu besar" / rate limit terus
# os.environ["LLM_MODEL"] = "openai/gpt-oss-20b"   # aktifkan jika model 120b tidak tersedia / kuota harian habis

## 2b. Cek koneksi Groq
Harus muncul `tersedia: True` dan `Tes panggilan: OK`. Jika model tidak tersedia, pilih salah satu dari daftar yang dicetak lalu isi `LLM_MODEL` di sel 2, restart session, jalankan ulang.

In [ ]:
import rag_gap_matrix as R
R.check_groq()

## 3. Unduh PDF (open access)
Jika ada yang GAGAL, unduh manual dari link DOI lalu upload ke folder `papers/` dengan nama `P0x.pdf`.

In [ ]:
import pandas as pd, rag_gap_matrix as R
papers = pd.read_csv(R.PAPERS_CSV)
R.download_pdfs(papers)
!ls -la papers

## 4. Cek tahap 1–2: ekstraksi & chunking satu paper

In [ ]:
pages = R.extract_pages(R.PDF_DIR / "P01.pdf")
chunks = R.chunk_pages("P01", pages)
print(len(pages), "halaman |", len(chunks), "chunk")
print(chunks[0].page, chunks[0].text[:500])

## 5. Jalankan pipeline penuh
±10 panggilan ekstraksi + 1 analisis gap. Jika kena rate limit free tier, skrip menunggu lalu mencoba lagi otomatis.

In [ ]:
res = R.run(download=False)
res["matrix"][["ID", "Problem", "Metode", "Temuan"]].head(10)

## 6. Evaluasi: latency, token, skor dukungan, kemiripan dengan acuan manual

In [ ]:
print(res["log"].describe()[["latency_s", "prompt_tokens", "completion_tokens"]])
print("\nSel yang perlu dicek manual:", res["flags"] or "tidak ada")
res["ref_cmp"]

In [ ]:
import json
print(json.dumps(res["gap"], ensure_ascii=False, indent=2))

## 7. Unduh hasil
`Literature_GAP_Matrix_RAG_Kesehatan.docx` → unggah ke Kulino. `gap_matrix.xlsx` berisi bukti retrieval per sel.

In [ ]:
from google.colab import files
for f in ["Literature_GAP_Matrix_RAG_Kesehatan.docx", "gap_matrix.xlsx", "gap_matrix.md"]:
    files.download(f"outputs/{f}")